# Session 3 — Surface Data, Pressure Coefficient, and Aerodynamic Loads

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Forces and moments come from integrating surface pressure and wall shear over a body — not from a single probe value. This session builds that pipeline: surface pressure to pressure coefficient, pressure coefficient to integrated loads, and pressure loads combined with a viscous contribution into total force coefficients.

## Learning outcomes
- Compute the pressure coefficient $C_p$ from surface pressure data.
- Integrate surface pressure into pressure-force contributions to lift and drag.
- Add a synthetic viscous (wall-shear) contribution and combine it with the pressure contribution.
- Sanity-check integrated loads against expected physical ranges.

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
print("Environment ready.")

## 1. Synthetic surface data

We generate pressure and wall-shear stress around a cylinder-like surface, parameterized by angle $\theta$ (0° at the front stagnation point, 180° at the rear). This mirrors a Fluent "wall" surface export with `x, y, pressure, wall-shear-x, wall-shear-y`. A small angle of attack (`alpha_deg`) shifts the separation region off of dead-rear-center, which is what gives this body a genuine (non-noise) net lift — a purely fore-aft-symmetric body at zero angle of attack should integrate to essentially zero lift.

In [ ]:
n_pts = 200
theta_deg = np.linspace(0, 360, n_pts, endpoint=False)
theta_rad = np.deg2rad(theta_deg)

R_m = 0.05
x_m = R_m * np.cos(theta_rad)
y_m = R_m * np.sin(theta_rad)

U_inf_mps = 15.0
rho_kgpm3 = 1.225
mu_Pas = 1.8e-5
p_inf_Pa = 101325.0
alpha_deg = 6.0  # small angle of attack: shifts the separation region so the wake is genuinely asymmetric

# Ideal-flow-like Cp with an angle-of-attack-shifted, separated-wake correction, plus noise
Cp_ideal = 1 - 4 * np.sin(theta_rad) ** 2
separation_mask = (theta_deg > 100 + alpha_deg) & (theta_deg < 260 + alpha_deg)
Cp_true = np.where(separation_mask, -0.9, Cp_ideal)
Cp_measured = Cp_true + rng.normal(0, 0.03, n_pts)

pressure_Pa = p_inf_Pa + Cp_measured * 0.5 * rho_kgpm3 * U_inf_mps ** 2
wall_shear_Pa = 0.6 * np.abs(np.sin(theta_rad)) * np.exp(-((theta_deg - 0) % 360) / 200) + rng.normal(0, 0.01, n_pts)

surface = pd.DataFrame({"theta_deg": theta_deg, "x_m": x_m, "y_m": y_m,
                         "pressure_Pa": pressure_Pa, "wall_shear_Pa": wall_shear_Pa})
surface.head()

## 2. Pressure coefficient

$$C_p = \frac{p - p_\infty}{\tfrac{1}{2}\rho U_\infty^2}$$

$C_p$ nondimensionalizes surface pressure against the freestream dynamic pressure, which lets you compare surfaces of different size or freestream speed on the same plot.

In [ ]:
q_inf_Pa = 0.5 * rho_kgpm3 * U_inf_mps ** 2
surface["Cp"] = (surface["pressure_Pa"] - p_inf_Pa) / q_inf_Pa

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(surface["theta_deg"], surface["Cp"])
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xlabel("theta (deg)")
ax.set_ylabel("Cp (-)")
ax.set_title(f"Surface pressure coefficient, U_inf = {U_inf_mps:.1f} m/s")
ax.invert_yaxis()  # aerodynamic convention: suction (negative Cp) plotted upward
plt.tight_layout()
plt.show()

print(f"Cp range: [{surface['Cp'].min():.2f}, {surface['Cp'].max():.2f}]  "
      f"(theoretical max at stagnation point is Cp = 1.0)")

### Checkpoint 1 — Sanity-check Cp
The stagnation point (front of the body) should have $C_p$ close to +1.0, and $C_p$ can legitimately go below -1 in a separated wake, but a value far outside roughly [-3, 1.2] usually signals a data or reference-value error. Confirm the stagnation value in this dataset and explain, in one sentence, what you would check first if it were badly off from 1.0.

## 3. Integrating pressure into loads

Pressure acts normal to the surface. Integrating the normal-direction component of pressure force around the closed surface gives the pressure contribution to lift and drag. For a discretized surface, approximate the integral as a sum over panels of length $ds \approx R\,\Delta\theta$.

In [ ]:
d_theta_rad = np.deg2rad(360 / n_pts)
ds_m = R_m * d_theta_rad  # arc length per panel (per unit span)

# Outward normal at each point on a circle points radially outward
normal_x = np.cos(theta_rad)
normal_y = np.sin(theta_rad)

# Pressure force per unit span = -p * normal * ds  (pressure pushes inward on the body)
pressure_force_x_Npm = np.sum(-surface["pressure_Pa"].values * normal_x * ds_m)
pressure_force_y_Npm = np.sum(-surface["pressure_Pa"].values * normal_y * ds_m)

# Reference the *gauge* pressure so integration stays well-conditioned (see Checkpoint 2)
pressure_force_x_gauge_Npm = np.sum(-(surface["pressure_Pa"].values - p_inf_Pa) * normal_x * ds_m)
pressure_force_y_gauge_Npm = np.sum(-(surface["pressure_Pa"].values - p_inf_Pa) * normal_y * ds_m)

print(f"Pressure drag (x): {pressure_force_x_gauge_Npm:8.3f} N/m")
print(f"Pressure lift (y): {pressure_force_y_gauge_Npm:8.3f} N/m")
print("Quality check: for a body with fore-aft symmetry and no separation, lift should be near zero — "
      f"here the {alpha_deg:.0f} deg angle-of-attack shift in the separation region breaks that symmetry, "
      "giving a genuine nonzero pressure lift (distinct from the noise-level residual you would see with alpha_deg = 0).")

### Checkpoint 2 — Why gauge pressure, and when raw pressure actually fails
For a properly closed, watertight surface, $\oint p_\infty \hat{n}\,dS = 0$ exactly — the outward normals cancel around the loop, so a *uniform* freestream pressure contributes zero net force whether or not you subtract $p_\infty$ first. Confirm this: integrate `pressure_Pa` directly (without subtracting `p_inf_Pa`) over the **full closed surface** using the noise-free `Cp_ideal` field (no separation). You should get a force at or near machine-precision zero — matching the gauge-pressure result, not contradicting it.

Now break the closed-surface assumption: repeat the same raw-pressure integration using only the **front-half arc** (`theta_deg < 180`) as a stand-in for an incomplete/non-watertight export (e.g., a solver "wall" zone missing a patch). Because that partial arc is not closed, $\oint \hat{n}\,dS \neq 0$ over it, so integrating *raw* pressure now picks up a large spurious force purely from $p_\infty$ leaking through the open boundary — while the gauge-pressure integral on the same open arc is far smaller. That is the real, practical reason to work in gauge pressure: it keeps physically small forces numerically well-conditioned and protects you against exactly this kind of open/non-watertight-surface artifact, even though on a genuinely closed surface the raw and gauge integrals agree.

In [ ]:
# TODO:
# 1. Integrate pressure_Pa directly (WITHOUT subtracting p_inf_Pa) over the FULL closed surface,
#    using Cp_ideal (no separation, no noise) to build a noise-free pressure field. Confirm the
#    result is ~0 (machine precision), matching what you get by subtracting p_inf_Pa first.
#
# 2. Repeat using only the front-half arc (theta_deg < 180) as a stand-in for an incomplete/open
#    surface export. Compare the raw-pressure vs. gauge-pressure integrated force on this open arc
#    and explain, in one sentence, why they now disagree when they did not on the closed surface.


## 4. Viscous contribution and total coefficients

Total drag = pressure (form) drag + viscous (friction) drag. Wall shear stress acts tangential to the surface; its streamwise component contributes to viscous drag.

In [ ]:
tangent_x = -np.sin(theta_rad)
tangent_y = np.cos(theta_rad)

viscous_force_x_Npm = np.sum(surface["wall_shear_Pa"].values * tangent_x * ds_m)
viscous_force_y_Npm = np.sum(surface["wall_shear_Pa"].values * tangent_y * ds_m)

total_drag_Npm = pressure_force_x_gauge_Npm + viscous_force_x_Npm
total_lift_Npm = pressure_force_y_gauge_Npm + viscous_force_y_Npm

chord_m = 2 * R_m
Cd = total_drag_Npm / (q_inf_Pa * chord_m)
Cl = total_lift_Npm / (q_inf_Pa * chord_m)

print(f"Pressure drag: {pressure_force_x_gauge_Npm:7.4f} N/m   Viscous drag: {viscous_force_x_Npm:7.4f} N/m")
print(f"Total drag:    {total_drag_Npm:7.4f} N/m  ->  Cd = {Cd:.4f}")
print(f"Total lift:    {total_lift_Npm:7.4f} N/m  ->  Cl = {Cl:.4f}")
print(f"Viscous fraction of drag: {100*viscous_force_x_Npm/total_drag_Npm:.1f}% "
      f"— {'physically reasonable for a bluff body (form-drag dominated)' if viscous_force_x_Npm/total_drag_Npm < 0.3 else 'check wall-shear magnitude'}")

### Checkpoint 3 — Report the loads
Following the course's engineering standard, write a two-sentence result statement for $C_d$ that includes: value, reference (chord and $U_\infty$ used), a numerical-quality check (e.g., panel-count sensitivity — would doubling `n_pts` change this by more than 1%?), and one limitation (this is a 2D per-unit-span estimate on synthetic, not solver, data).

## Graduate/Advanced Extension
Repeat the pressure-drag integration with `n_pts` reduced to 40 and increased to 800. Plot $C_d$ vs. `n_pts` on a semilog-x axis. At what panel count does the integrated drag stop changing by more than 0.5%? Relate this to the mesh-independence exercise from Session 1 — panel-count independence for surface integration is the same idea applied to post-processing rather than to the solver mesh.

## Exit ticket
In three sentences: state the difference between $C_p$ and $C_d$ in your own words, describe one physical check you now know to run on integrated loads before reporting them, and name one real geometry from your own work where separating pressure and viscous drag would matter.

**Next:** Session 4 moves from steady surface loads to transient force histories and their frequency content.